# CaQR 162-Circuit Benchmark Campaign - Qiskit 0.45

The **162 benchmark circuits were selected before inspecting the campaign results**: 38 were previously assigned to training, 12 to validation, 110 have no previous split assignment, and 2 are reserved for final testing.

This notebook runs the **actual CaQR generator** and evaluates the historical Ridge-based repair with SABRE routing. It retains every circuit in the screening report, including cases with no qubit reuse, no valid repair neighbor, or execution errors. Circuits are **never selected or excluded based on positive gain**.

The first stage screens 160 development circuits. The two reserved final-test circuits remain locked.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
import pandas as pd
import qiskit
from IPython.display import display

REPAIR_ROOT = Path(r'C:\Users\ASUS\Documents\Quantum computing\qubit-reuse-repair')
CAQR_ROOT = Path(r'C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main')
OUT_DIR = REPAIR_ROOT / 'caqr_162_results'
SCRIPT = REPAIR_ROOT / 'scripts' / 'caqr_campaign.py'

assert SCRIPT.is_file(), f'Extract the add-on ZIP inside REPAIR_ROOT: {SCRIPT}'
assert (REPAIR_ROOT / 'pilot_models' / 'swap_count_one_step_TRAIN_ONLY.json').is_file()
assert (CAQR_ROOT / 'main.py').is_file()
assert qiskit.__version__.startswith('0.45.'), qiskit.__version__
print('Python:',sys.executable,'Qiskit:',qiskit.__version__)
print('CaQR:',CAQR_ROOT,'Repair:',REPAIR_ROOT)

In [ ]:
def run_campaign(action, *options):
    env=dict(os.environ)
    env['PYTHONPATH']=str(REPAIR_ROOT / 'src') + os.pathsep + env.get('PYTHONPATH','')
    cmd=[sys.executable,str(SCRIPT),action,'--out-dir',str(OUT_DIR),*map(str,options)]
    print('RUN:', ' '.join(cmd),flush=True)
    done=subprocess.run(cmd,cwd=str(REPAIR_ROOT),env=env)
    if done.returncode!=0:
        raise RuntimeError(f'Campaign {action} failed, return code {done.returncode}')

run_campaign('roster','--caqr-root',CAQR_ROOT)
roster_df=pd.read_csv(OUT_DIR/'roster_162.csv')
print('Total:',len(roster_df))
display(roster_df.groupby('previous_split').size().rename('circuits').to_frame())
assert len(roster_df)==162

## Phase 1 - Screen circuits using the actual CaQR generator

Start with **20 previously unscreened circuits** per batch. After confirming that the first batch works, set `SCREEN_BATCHES = 8` to process approximately 160 development circuits, skipping completed cases. To process all remaining circuits in a single call, set `SCREEN_BATCH_SIZE = 0`.

Each circuit's outcome is saved immediately. You can rerun these cells after an interruption without repeating completed work. **The reserved final-test circuits are excluded.**


In [ ]:
SCREEN_BATCH_SIZE = 20
SCREEN_BATCHES = 1  # set 8 for all ~160 development cases; resume skips finished
for batch in range(SCREEN_BATCHES):
    print(f'\nScreening batch {batch+1}/{SCREEN_BATCHES}')
    run_campaign('screen','--caqr-root',CAQR_ROOT,'--batch-size',SCREEN_BATCH_SIZE,
                 '--timeout',300,'--attempts',150)

In [ ]:
screen=pd.read_csv(OUT_DIR/'screening_162.csv')
display(screen.groupby('status',dropna=False).size().rename('n_circuits').to_frame())
print('Ready for Ridge/Routing:', int((screen['status']=='ready').sum()))
print('Not screened:',int((screen['status']=='not_screened').sum()))
view=['circuit','previous_split','family_heuristic','active_qubits','active_width',
      'reuse_edges','valid_one_step_neighbors','status','error']
display(screen[view].head(30))

## Phase 2 - Ridge repair and routing for eligible circuits

Each evaluation job represents **one circuit x one objective x one routing seed**. The original and proposed plans use the same dynamic-circuit materializer, SABRE routing settings, and main routing seed. The `Original Best-of-2` control uses a separate second seed.

The two **one-step TRAIN_ONLY** Ridge checkpoints are reused as fixed models; no retraining is performed. Three routing seeds are prescribed: `200, 211, 223`. Keep the same seeds and settings when resuming the campaign.

First run `EVALUATION_BATCH_SIZE = 4` and inspect the reports, then increase it to 20 or 40 jobs per batch. A complete development campaign can involve up to **960 circuit-objective-seed jobs** (160 circuits x 2 objectives x 3 seeds); in practice, only eligible circuits are routed.


In [ ]:
EVALUATION_BATCH_SIZE = 4  # pilot: 4 jobs; then 20 or 40; 0 = all remaining
EVALUATION_BATCHES = 1
for batch in range(EVALUATION_BATCHES):
    print(f'\nRouting batch {batch+1}/{EVALUATION_BATCHES}')
    run_campaign('evaluate','--batch-size',EVALUATION_BATCH_SIZE,
                 '--seeds','200,211,223','--topology','heavy_hex7',
                 '--budget',120,'--max-depth',5,'--routing-timeout',420)

In [ ]:
run_campaign('summary')
summary=json.loads((OUT_DIR/'summary_162.json').read_text(encoding='utf-8'))
print('Screening status:',summary['screen_counts'])
print('Routing job status:',summary['routing_job_counts'])
display(pd.DataFrame(summary['groups']))

## Phase 3 - Per-circuit results, including unsuccessful cases

The table below contains exactly **324 rows**: 162 circuits x 2 separate optimization objectives. `mean_guard_gain_pct` is available only when at least one successful routing seed yields a defined percentage. The `status` and `n_seeds_succeeded` fields indicate whether the planned evaluation is complete.

For any primary published aggregate, include only cases with all prescribed seeds completed, and report failures and coverage separately. Do not silently remove failed circuits from the screening denominator.


In [ ]:
results=pd.read_csv(OUT_DIR/'per_circuit_objective.csv')
assert len(results)==324, len(results)
cols=['circuit','previous_split','family_heuristic','objective','screen_status',
      'n_seeds_succeeded','n_seeds_requested','mean_guard_gain_pct',
      'mean_proposed_gain_abs','mean_guard_minus_best2_abs','status']
display(results[cols].sort_values(['objective','circuit']))
print('Results CSV:',OUT_DIR/'per_circuit_objective.csv')

In [ ]:
# Strict completed-case analysis: do not mix observed partial seeds or reserved test
complete=results[(results['n_seeds_succeeded']==results['n_seeds_requested']) &
                 (results['n_percent_defined']==results['n_seeds_requested']) &
                 (results['previous_split']!='final_test_reserved')]
print('Completed circuit-objective pairs:',len(complete))
if len(complete):
    display(complete.groupby(['objective','previous_split']).agg(
        n=('circuit','count'),
        mean_guard_gain_pct=('mean_guard_gain_pct','mean'),
        median_guard_gain_pct=('mean_guard_gain_pct','median'),
        mean_benefit_vs_best2=('mean_guard_minus_best2_abs','mean')).round(3))
else:
    print('No fully evaluated cases yet; continue the evaluation batches.')

### Reporting rules for the abstract and the paper

- Do **not** say that 160 circuits were evaluated unless screening and/or routing actually covered that many circuits. Report the exact completed count for each stage.
- The Ridge checkpoints are **TRAIN_ONLY** and may overlap in circuits or circuit families with this campaign. Do **not** describe all results as independent external validation.
- The final guard guarantees non-worsening of the selected metric **only for the specific observed matched routing comparison**. It is not evidence that the predictor always ranks plans correctly or that gains generalize across seeds.
- **CaQR generates the original reuse plans**, but the original and repaired plans are both compiled using the project's **shared materializer**. This is not a comparison of native CaQR compiled output against native CaQR compiled output.
- Report **SWAP count** and **routed depth** as separate objectives, never as a weighted composite.
- Preserve the two reserved final-test cases until the experimental protocol and model are frozen.
